## Manejo de datos faltantes

### Situación 1. Observaciónes faltantes
    * Las marcas de tiempo van a estar completas pero van a haber datos faltantes en las columnas

### Situación 2. Marcas de tiempo faltantes
    * Las columnas van a estar completas pero los índices son los que tienen valores faltantes etiquetadas como NaT

### Situación 3. Marcas de tiempo ocultas
    * Las columnas van a estar completas pero los índices son los que tienen valores faltantes pero no van a estar etiquetadas como NaT

### Situación 4. Faltan observaciónes y marcas de tiempo
    * Observaciónes faltantes marcadas como NaN y indices faltantes marcados como NaT  

### Situación 1. Observaciónes faltantes

In [9]:
import pandas as pd

co2 = pd.read_csv('datos/co2_faltantes.csv',
                    parse_dates=['año'],
                    index_col = 'año')
co2.head()

,co2
año,
1750-01-01,0.0125
1760-01-01,0.0128
1770-01-01,NaN
1780-01-01,0.0169
1790-01-01,0.0206


In [11]:
co2.isna().sum()

co2    26
dtype: int64

In [12]:
idx_faltante = co2[co2['co2'].isna()].index
print(idx_faltante)

DatetimeIndex(['1770-01-01', '1927-01-01', '1928-01-01', '1929-01-01',
               '1930-01-01', '1931-01-01', '1932-01-01', '1933-01-01',
               '1934-01-01', '1935-01-01', '1936-01-01', '1937-01-01',
               '1974-01-01', '1975-01-01', '1976-01-01', '1977-01-01',
               '1978-01-01', '1979-01-01', '1980-01-01', '1981-01-01',
               '1982-01-01', '1983-01-01', '1984-01-01', '1985-01-01',
               '1986-01-01', '1987-01-01'],
              dtype='datetime64[us]', name='año', freq=None)


### Situación 2. Marcas de tiempo faltantes

In [ ]:
clicks = pd.read_csv('datos/clicks_faltantes_marcas.csv',
                        parse_dates = ['fecha'],
                        index_col = 'fecha')
clicks.head() # Se genera un NaT

/tmp/ipykernel_188606/2517434643.py:1: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  clicks = pd.read_csv('datos/clicks_faltantes_marcas.csv',


,precio,ubicación,clicks
fecha,,,
2008-04-01,43.155647,2,18784.0
2008-04-02,43.079056,1,24738.0
NaT,43.842609,2,15209.0
2008-04-04,43.382794,2,14320.0
2008-04-05,43.941176,1,11974.0


In [ ]:
clicks.isna().sum() # En las observaciónes

precio        0
ubicación     0
clicks       16
dtype: int64

In [ ]:
clicks.index.isna().sum() # En el índice

np.int64(2)

### Situación 3. Marcas de tiempo ocultas

In [ ]:
clicks = pd.read_csv('datos/clicks_faltantes_marcas_ocultas.csv',
                        parse_dates = ['fecha'],
                        index_col = 'fecha')
clicks.head()
# del 04 se pasa al 05 en el indice

/tmp/ipykernel_188606/2814900262.py:1: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  clicks = pd.read_csv('datos/clicks_faltantes_marcas_ocultas.csv',


,precio,ubicación,clicks
fecha,,,
2008-04-01,43.155647,2,18784.0
2008-04-02,43.079056,1,24738.0
2008-04-03,43.842609,2,15209.0
2008-04-05,43.941176,1,11974.0
2008-04-06,44.403936,1,11007.0


In [22]:
clicks.isna().sum()

precio        0
ubicación     0
clicks       16
dtype: int64

In [ ]:
clicks.index.isna().sum() 
# A simple vista no ocurre nada extraño en los indices

np.int64(0)

Tenemos que observar que la serie de tiempo tiene una frecuencia diaria, pero en realidad hay valores faltantes que hacen que no se cumpla, se tiene que implementar:

    * Un nuevo índice con rango ideal de fechas
    * Usar "diference" para encontrar las diferencias de esos rangos

In [25]:
rango_fechas = pd.date_range(start = clicks.index.min(), end = clicks.index.max(), freq = 'D')
rango_fechas

DatetimeIndex(['2008-04-01', '2008-04-02', '2008-04-03', '2008-04-04',
               '2008-04-05', '2008-04-06', '2008-04-07', '2008-04-08',
               '2008-04-09', '2008-04-10',
               ...
               '2008-08-04', '2008-08-05', '2008-08-06', '2008-08-07',
               '2008-08-08', '2008-08-09', '2008-08-10', '2008-08-11',
               '2008-08-12', '2008-08-13'],
              dtype='datetime64[us]', length=135, freq='D')

In [ ]:
rango_fechas.difference(clicks.index) # Estas no eran visibles porque estaban ocultas

DatetimeIndex(['2008-04-04', '2008-04-07', '2008-04-09'], dtype='datetime64[us]', freq=None)


### Situación 4. Faltan observaciónes y marcas de tiempo

In [28]:
clicks = pd.read_csv('datos/clicks_faltantes_multiples.csv',
                        parse_dates = ['fecha'],
                        index_col = 'fecha')
clicks.head()

/tmp/ipykernel_188606/3495471139.py:1: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  clicks = pd.read_csv('datos/clicks_faltantes_multiples.csv',


,precio,ubicación,clicks
fecha,,,
2008-04-01,43.155647,2.0,18784
2008-04-02,43.079056,1.0,24738
NaT,43.842609,NaN,15209
NaT,NaN,1.0,14018
NaT,43.941176,1.0,11974


In [29]:
clicks.isna().sum()

precio        1
ubicación     1
clicks       14
dtype: int64

In [32]:
NaN_clicks = clicks[clicks.isna()].index
NaN_clicks

DatetimeIndex(['2008-04-01', '2008-04-02',        'NaT',        'NaT',
                      'NaT',        'NaT', '2008-04-07', '2008-04-08',
               '2008-04-09', '2008-04-10',
               ...
               '2008-08-04', '2008-08-05', '2008-08-06', '2008-08-07',
               '2008-08-08', '2008-08-09', '2008-08-10', '2008-08-11',
               '2008-08-12', '2008-08-13'],
              dtype='datetime64[us]', name='fecha', length=135, freq=None)

In [34]:
clicks.index.isna().sum()

np.int64(4)